# Dynamic RAG Hybrid Retrieval

## Dependencies

In [ ]:
%pip install datasets
%pip install langchain-core
%pip install langchain-openai
%pip install langchain-pinecone
%pip install matplotlib
%pip install nltk
%pip install pandas
%pip install ragas
%pip install rank_bm25
%pip install scikit-learn
%pip install spacy
%pip install tqdm
%pip install transformers
!python -m spacy download en_core_web_sm

## Imports

In [ ]:
from concurrent.futures import ThreadPoolExecutor  # For parallel execution of tasks
from datasets import load_dataset, Dataset  # For loading datasets
from functools import lru_cache  # For caching results of expensive functions
from IPython.display import display, Markdown  # For displaying results in Jupyter
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from nltk.corpus import stopwords  # For removing stop words in text processing
from nltk.stem import PorterStemmer  # For stemming words
from pinecone import Pinecone, ServerlessSpec  # For Pinecone vector store
from ragas import evaluate
from ragas.evaluation import EvaluationResult
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    answer_relevancy,
    answer_similarity,
    answer_correctness,
    FactualCorrectness,
)  # For evaluation metrics
from ragas.run_config import RunConfig  # For configuring evaluation runs
from rank_bm25 import BM25Okapi  # For BM25 retrieval
from scipy.stats import entropy, wilcoxon  # For statistical analysis
from tqdm.auto import tqdm  # For progress bars
from transformers import (
    AutoTokenizer,
)  # For tokenization and embedding generation using transformer models
from typing import Tuple, List

import matplotlib.pyplot as plt
import nltk
import numpy as np
import os
import pandas as pd
import re
import requests
import spacy  # For NLP tasks like tokenization and entity recognition
import threading
import time

## Constants

### API Keys

In [ ]:
DEEP_INFRA_API_KEY = "SET_YOUR_API_KEY"
os.environ["OPENAI_API_KEY"] = "SET_YOUR_API_KEY"
PINECONE_API_KEY = "SET_YOUR_API_KEY"

### Dataset

In [ ]:
MAX_QUESTIONS = 100  # Limit for testing

### DeepInfra

In [5]:
DEEP_INFRA_API_BASE = "https://api.deepinfra.com/v1/openai"

### Dynamic Threshold

In [6]:
KAPPA = 2.0  # Scaling factor for how much entropy influences the threshold
H_MID = 1.6
T_MIN = 0.3
T_MAX = 0.6

### LLM

In [7]:
ALPHA = 0.7  # Weighting factor for retrieval uncertainty
MAX_TOKENS = 256  # Max tokens for retrieval context
TOP_K = 10  # Number of top results to retrieve

### Models

In [8]:
EMBEDED_MODEL = "text-embedding-3-small"  # Embedding model to use

### Pinecone

In [9]:
PINECONE_ENVIRONMENT = "us-east-1"
PINECONE_INDEX_NAME = "dynamic-rag-hybrid-retrieval"
PINECONE_EMBED_DIMENSION = 1536

### SoftMax

In [10]:
TEMPERATURE = 1.0

## Initialization

In [ ]:
# Initialize Spacy for Named Entity Recognition (NER)
nlp = spacy.load("en_core_web_sm")

# Initialize Tokenizer (GPT-2 is a safe default)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

run_config = RunConfig(max_workers=16, timeout=90, max_retries=3)
wikidata_lock = threading.Lock()  # Lock for synchronizing access to Wikidata API

stemmer = PorterStemmer()  # For stemming words in text processing

nltk.download("stopwords")
stop_words = set(
    stopwords.words("english")
)  # For removing stop words in text processing

## Helper Functions

In [12]:
def count_tokens(text: str) -> int:
    """
    Calculates the number of tokens in a given text string.

    Args:
        text: The text string to tokenize.

    Returns:
        The number of tokens in the text.
    """

    return len(tokenizer.encode(text))

In [13]:
def min_max_normalize(scores: np.ndarray) -> np.ndarray:
    """
    Normalizes an array of scores using min-max normalization.

    Args:
        scores: A numpy array of scores to be normalized.

    Returns:
        A numpy array of min-max normalized scores, where each value is scaled to the range [0, 1].
    """

    if scores is None or len(scores) == 0:
        return np.array([])  # Return an empty array if there are no scores

    min_score = np.min(scores)
    max_score = np.max(scores)

    if min_score == max_score:
        return np.ones_like(scores) / len(
            scores
        )  # If all scores are the same, return a uniform distribution

    return (scores - min_score) / (max_score - min_score)

In [14]:
def softmax_normalize_with_temperature(
    scores: np.ndarray, temperature: float = 1.0
) -> np.ndarray:
    """
    Normalizes an array of scores using the softmax function.

    Args:
        scores: A numpy array of scores to be normalized.
        temperature: The temperature parameter for softmax normalization. Default is 1.0, which give standard softmax.

    Returns:
        A numpy array of softmax-normalized scores, where each value is transformed to the range (0, 1) and the sum of all values equals 1.
    """

    scores = scores / temperature
    exp_scores = np.exp(scores - np.max(scores))

    return exp_scores / np.sum(exp_scores)

In [15]:
def compute_entropy(probabilities: np.ndarray) -> float:
    """
    Computes the entropy of a probability distribution.

    Args:
        probabilities: A numpy array representing a probability distribution (values should sum to 1).

    Returns:
        The entropy of the distribution, which is a measure of uncertainty. Higher values indicate more uncertainty.
    """

    p = probabilities[
        probabilities > 1e-8
    ]  # Filter out very small probabilities to avoid log(0)

    return -np.sum(p * np.log(p))  # Compute entropy using the formula H

In [16]:
def calculate_dynamic_threshold(entropy: float) -> float:
    """
    Calculates a dynamic threshold based on the entropy of the retrieval scores.

    Args:
        entropy: The entropy of the retrieval scores, which indicates the uncertainty of the retrieval results.

    Returns:
        The dynamic threshold, which is a value between T_MIN and T_MAX based on the entropy.
    """

    sigmoid = 1 / (1 + np.exp(KAPPA * (entropy - H_MID)))

    # If entropy is low, the threshold is close to T_MIN
    # If entropy is high, the threshold is close to T_MAX
    return T_MIN + (T_MAX - T_MIN) / sigmoid

## Judge LLM

In [ ]:
judge_model = "gpt-4o-mini"

judge_llm = ChatOpenAI(
    model_name=judge_model,
    openai_api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,  # 0 is the most deterministic
)

judge = LangchainLLMWrapper(judge_llm, bypass_n=True)

answer_relevancy.llm = judge
answer_similarity.llm = judge
answer_correctness.llm = judge
factual_f1 = FactualCorrectness(llm=judge, mode="f1")

## Wikidata

In [18]:
def filter_wikidata_results(bindings: List[dict]) -> List[str]:
    """
    Filters and formats the results returned from a Wikidata SPARQL query.

    Args:
        bindings: A list of dictionaries representing the raw results from the SPARQL query.

    Returns:
        A list of formatted strings containing the property labels and their corresponding object labels, sorted by rank.
    """

    filtered_facts = []

    deprecated_uri = "https://wikiba.se/ontology#DeprecatedRank"

    for binding in bindings:
        rank = binding.get("rank", {}).get("value", "")
        prop = binding.get("pLabel", {}).get("value", "")
        val = binding.get("oLabel", {}).get("value", "")

        if rank == deprecated_uri:
            continue  # Skip deprecated facts

        noise_list = [
            "image",
            "logo",
            "id",
            "instance of",
            "coordinate",
            "audio",
            "video",
        ]
        if any(noise in prop.lower() for noise in noise_list):
            continue  # Skip noisy properties

        filtered_facts.append(f"{prop}: {val}")

    return list(set(filtered_facts))  # Remove duplicates

In [19]:
@lru_cache(maxsize=10000)
def query_wikidata(entity: str) -> tuple:
    """
    Queries Wikidata for information about a given entity.

    Args:
        entity: The entity to query (e.g., "Albert Einstein").

    Returns:
        A tuple containing information about the entity, or an empty tuple
        if no information is found.
    """

    clean_entity = entity.replace("\n", " ").strip()  # Clean the entity name

    url = "https://query.wikidata.org/sparql"
    query = f"""
    SELECT DISTINCT ?pLabel ?oLabel WHERE {{
        ?item rdfs:label "{clean_entity}"@en.
        
        # Fast direct-path lookup
        ?item ?p ?o.
        ?property wikibase:directClaim ?p.
        
        # Get English labels for the property and the value
        SERVICE wikibase:label {{ 
            bd:serviceParam wikibase:language "en". 
            ?property rdfs:label ?pLabel.
            ?o rdfs:label ?oLabel.
        }}
        
        # Ignore common administrative/metadata properties to save time
        FILTER(!STRSTARTS(STR(?pLabel), "modified"))
        FILTER(!STRSTARTS(STR(?pLabel), "described at"))
    }}
    ORDER BY ASC(?pLabel)
    LIMIT 3
    """
    headers = {
        "Accept": "application/json",
        "User-Agent": "DynamicRAGHybridRetrievalBot/1.0",
    }

    with wikidata_lock:  # Ensure that only one thread queries Wikidata at a time
        # Force a delay between requests to respect Wikidata's rate limits (under 2 requests per second)
        time.sleep(0.6)

        try:
            response = requests.get(url, params={"query": query}, headers=headers)

            if response.status_code == 200:
                data = response.json()
                bindings = data.get("results", {}).get("bindings", [])

                return filter_wikidata_results(bindings)
            elif response.status_code == 429:
                print("[Wikidata] Rate limit exceeded. Retrying after a short delay...")
                time.sleep(5)  # Wait before retrying

                return query_wikidata(entity)  # Retry the query
        except requests.exceptions.RequestException as e:
            print(f"[Wikidata] {e}")
            return []

In [20]:
def enrich_text_with_wikidata(text: str) -> str:
    """
    Identifies named entities in the given text and retrieves corresponding
    information from Wikidata.

    Args:
        text: The text to enrich.

    Returns:
        A string containing Wikidata information about the entities found in the text.
        Returns an empty string if no relevant entities are found.
    """

    doc = nlp(text)
    unique_entities = list(
        set(
            [
                ent.text
                for ent in doc.ents
                if ent.label_ in ["ORG", "PERSON", "GPE", "PRODUCT"]
            ]
        )
    )  # Filter for relevant entity types

    if not unique_entities:
        return ""

    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(query_wikidata, unique_entities))

    wikidata_knowledge = []
    for entity, facts in zip(unique_entities, results):
        if facts:
            wikidata_knowledge.append(f"{entity}: {'; '.join(facts)}")

    return "\n".join(wikidata_knowledge) if wikidata_knowledge else ""

## Dataset

### Corpus

This dataset contains chunked extracts (of ~300 tokens) from papers related to (and including) the Llama 2 research paper. Related papers were identified by following a trail of references, extracting those papers with the arxiv-bot package, and repeating.

In [ ]:
dataset = load_dataset("jamescalam/llama-2-arxiv-papers-chunked", split="train")
data = dataset.to_pandas()
corpus = [x["chunk"] for _, x in data.iterrows()]

print("Corpus Length:", len(corpus))
print(corpus[0])

### Evaluation Dataset

This ML Q&A dataset contains 43,713 samples, where each includes three fields - question, context(title + abstract) and answer. It is created based on the original dataset aalksii/ml-arxiv-papers, which contains the titles and abstracts of ML ArXiv papers.

In [ ]:
eval_dataset = load_dataset("hanyueshf/ml-arxiv-papers-qa", split="train")
eval_df = pd.DataFrame(eval_dataset)
eval_df = eval_df.drop(columns=["id", "context"])
eval_df = eval_df.rename(columns={"answer": "ground_truth"})

print("Evaluation Dataset Shape:", eval_df.shape)
print(eval_df.head())

In [23]:
eval_df_limited = eval_df.head(100)

### Leakage Check

In [24]:
def check_leakage(corpus: List[str], eval_df: pd.DataFrame) -> None:
    """
    Checks for potential leakage in the evaluation dataset.

    Args:
        corpus: A list of strings representing the corpus of documents.
        eval_df: A pandas DataFrame containing the evaluation dataset.
    """

    matches = 0

    for idx, row in eval_df.head(100).iterrows():
        if row["ground_truth"].lower() in " ".join(corpus).lower():
            print(f"Potential leakage at index {idx}")
            matches += 1

    if matches == 0:
        print("No leakage detected.")
    else:
        print(f"{matches} potential leaks detected.")

In [ ]:
check_leakage(corpus, eval_df)

## Model Embedding

In [26]:
embed_model = OpenAIEmbeddings(
    model=EMBEDED_MODEL, openai_api_key=os.environ["OPENAI_API_KEY"]
)

## BM25

### Functions

In [27]:
def tokenize(text: str) -> List[str]:
    """
    Tokenizes the input text using the GPT-2 tokenizer.

    Args:
        text: The text to tokenize.

    Returns:
        A list of tokens extracted from the input text.
    """

    if not text:
        return []  # Return an empty list if the input text is empty or None

    tokens = re.findall(r"[a-z0-9]+", text.lower())

    return [
        stemmer.stem(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]  # Remove stop words and short tokens

In [28]:
def initialize_bm25_index(corpus: List[str]) -> BM25Okapi:
    """
    Initializes a BM25Okapi index for sparse retrieval.

    Args:
        corpus: A list of text documents to index.

    Returns:
        A BM25Okapi index.
    """

    tokenized_corpus = [tokenize(doc) for doc in corpus]

    return BM25Okapi(tokenized_corpus)

### Initialization

In [29]:
bm25_index = initialize_bm25_index(corpus)

## Pinecone

### Functions

In [30]:
def initialize_pinecone(
    api_key: str, environment: str, index_name: str, embed_dimension: int
) -> Pinecone.Index:
    """
    Initializes the Pinecone index for vector storage.

    Returns:
        An instance of the Pinecone index ready for use.
    """

    pc = Pinecone(api_key=api_key)

    if index_name not in pc.list_indexes().names():
        pc.create_index(
            name=index_name,
            dimension=embed_dimension,
            metric="cosine",  # Or 'euclidean' based on your methodology
            spec=ServerlessSpec(cloud="aws", region=environment),
        )

    return pc.Index(index_name)

In [31]:
def upload_to_pinecone(
    index: Pinecone.Index, texts: List[str], embed_model: OpenAIEmbeddings
) -> None:
    """
    Uploads a list of texts to the Pinecone index after embedding them.

    Args:
        index: The Pinecone index to upload to.
        texts: A list of texts to upload.
        embed_model: The OpenAIEmbeddings model to use for embedding.
    """

    stats = index.describe_index_stats()
    current_vectors = stats.total_vector_count if stats else 0

    if current_vectors >= len(texts):
        print(
            f"Pinecone index already contains {current_vectors} vectors. Skipping upload."
        )
        return

    for i in tqdm(range(0, len(texts), 100), desc="Uploading to Pinecone"):
        batch = texts[i : i + 100]

        try:
            embeddings = embed_model.embed_documents(batch)

            vectors = []
            for j, (text, embedding) in enumerate(zip(batch, embeddings)):
                vector_id = f"doc_{i + j}"
                vectors.append((vector_id, embedding, {"text": text}))

            index.upsert(vectors)
            time.sleep(0.5)  # Sleep to avoid hitting rate limits
        except Exception as e:
            print(f"Error uploading batch starting at index {i}: {e}")
            continue

### Initialization

In [32]:
index = initialize_pinecone(
    PINECONE_API_KEY,
    PINECONE_ENVIRONMENT,
    PINECONE_INDEX_NAME,
    embed_dimension=PINECONE_EMBED_DIMENSION,
)

In [ ]:
upload_to_pinecone(index, corpus, embed_model)

## Retrieval

In [34]:
def retrieve_dense(index: Pinecone.Index, query: str, top_k: int) -> List[dict]:
    """
    Performs dense retrieval using Pinecone.

    Args:
        index: The Pinecone index to query.
        query: The query string.
        top_k: The number of results to retrieve.

    Returns:
        A list of dictionaries, where each dictionary represents a retrieved document
        and its associated score and metadata.
    """

    xq = embed_model.embed_query(query)  # Embed the query
    results = index.query(vector=xq, top_k=top_k, include_metadata=True)

    return results.get("matches", [])

In [35]:
def retrieve_sparse(
    bm25_index: BM25Okapi, query: str, corpus: List[str], top_k: int
) -> List[Tuple[int, float]]:
    """
    Performs sparse retrieval using BM25.

    Args:
        bm25_index: The BM25Okapi index.
        query: The query string.
        corpus: The list of documents.
        top_k: The number of top documents to return.

    Returns:
        A list of tuples, where each tuple contains the document index and its BM25 score.
    """

    tokenized_query = tokenize(query)
    doc_scores = bm25_index.get_scores(tokenized_query)
    # Get indices of the top_k documents
    top_indices = np.argsort(doc_scores)[::-1][:top_k]

    return [(i, doc_scores[i]) for i in top_indices]

In [36]:
def retrieve_hybrid(
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    query: str,
    top_k: int,
    alpha: float = 0.5,  # Weight for dense retrieval
    dynamic_threshold: bool = True,
) -> Tuple[List[dict], float]:
    """
    Performs hybrid retrieval by combining dense and sparse retrieval results.

    Args:
        index: The Pinecone index for dense retrieval.
        bm25_index: The BM25Okapi index for sparse retrieval.
        corpus: The list of documents.
        query: The query string.
        top_k: The number of results to retrieve from each method.
        alpha: Weighting factor for dense retrieval (1-alpha for sparse).
        dynamic_threshold: Whether to use dynamic thresholding.

    Returns:
        A tuple containing:
            - A list of retrieved documents, sorted by the combined score.
            - The threshold score used for filtering (if dynamic_threshold is True).
    """

    dense_results = retrieve_dense(index, query, top_k * 2)
    sparse_results = retrieve_sparse(bm25_index, query, corpus, top_k * 2)

    dense_raw = np.array([hit["score"] for hit in dense_results])
    sparse_raw = np.array([score for _, score in sparse_results])

    dense_norm = min_max_normalize(dense_raw)
    sparse_norm = min_max_normalize(sparse_raw)

    scores = {}

    for i, hit in enumerate(dense_results):
        text = hit["metadata"]["text"]
        scores.setdefault(text, [0.0, 0.0])
        scores[text][0] = float(dense_norm[i])  # Normalized dense score

    for i, (idx, _) in enumerate(sparse_results):
        text = corpus[idx]
        scores.setdefault(text, [0.0, 0.0])
        scores[text][1] = float(sparse_norm[i])  # Normalized sparse score

    combined = [
        {
            "text": text,
            "combined_score": alpha * dense
            + (1 - alpha) * sparse,  # Value between 0 and 1
        }
        for text, (dense, sparse) in scores.items()
    ]
    combined.sort(key=lambda x: x["combined_score"], reverse=True)
    combined = combined[:top_k]  # Keep only top_k results

    combined_scores = np.array([hit["combined_score"] for hit in combined])
    probabilities = softmax_normalize_with_temperature(combined_scores, TEMPERATURE)

    query_entropy = compute_entropy(probabilities)

    threshold = calculate_dynamic_threshold(query_entropy) if dynamic_threshold else 0.4
    score_threshold = threshold * combined_scores[0]  # Scale threshold by max score

    filtered = [
        hit["text"] for hit in combined if hit["combined_score"] >= score_threshold
    ]

    if not filtered:
        filtered = [combined[0]["text"]]  # Ensure at least one result is returned

    return filtered, threshold

## System Setup Validation

In [ ]:
def validate_system_setup(
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    eval_df: pd.DataFrame,
):
    """
    Validates the system setup for evaluation.

    Args:
        index: The Pinecone index.
        bm25_index: The BM25Okapi index.
        corpus: The list of documents.
        eval_df: The evaluation dataframe.

    Returns:
        A dictionary containing validation results.
    """

    results = {
        "openai_api": False,
        "pinecone": False,
        "bm25": False,
        "corpus": False,
        "eval_data": False,
        "retrieval": False,
    }

    # 1. OpenAI API Validation
    try:
        test_llm = ChatOpenAI(
            model_name="gpt-3.5-turbo",
            openai_api_key=os.environ["OPENAI_API_KEY"],
            max_tokens=10,
        )
        response = test_llm.invoke([HumanMessage(content="OK")])
        if response.content:
            results["openai_api"] = True
    except:
        pass

    # 2. Pinecone Validation
    try:
        stats = index.describe_index_stats()
        if stats.total_vector_count > 0:
            results["pinecone"] = True
    except:
        pass

    # 3. BM25 Validation
    try:
        test_scores = bm25_index.get_scores(tokenize("test"))
        if len(test_scores) > 0:
            results["bm25"] = True
    except:
        pass

    # 4. Corpus Validation
    try:
        if len(corpus) > 0 and len(corpus[0]) > 0:
            results["corpus"] = True
    except:
        pass

    # 5. Evaluation Data Validation
    try:
        if (
            len(eval_df) > 0
            and "question" in eval_df.columns
            and "ground_truth" in eval_df.columns
        ):
            results["eval_data"] = True
    except:
        pass

    # 6. Retrieval Validation
    try:
        test_q = "neural network"
        dense = retrieve_dense(index, test_q, top_k=1)
        sparse = retrieve_sparse(bm25_index, test_q, corpus, top_k=1)
        if len(dense) > 0 or len(sparse) > 0:
            results["retrieval"] = True
    except:
        pass

    return results


def check_setup():
    """
    Checks the system setup for evaluation.
    """

    print("\nSystem Setup Validation")
    print("-" * 40)

    results = validate_system_setup(index, bm25_index, corpus, eval_df)

    checks = [
        ("OpenAI API", results["openai_api"]),
        ("Pinecone Vector Store", results["pinecone"]),
        ("BM25 Index", results["bm25"]),
        ("Corpus", results["corpus"]),
        ("Evaluation Data", results["eval_data"]),
        ("Retrieval", results["retrieval"]),
    ]

    all_passed = True
    for name, status in checks:
        status_text = "PASS" if status else "FAIL"
        print(f"{name:20} : {status_text}")
        if not status:
            all_passed = False

    print("-" * 40)

    if all_passed:
        print("All systems ready for evaluation")
    else:
        print("Please fix failing components before proceeding")
        if not results["pinecone"]:
            print("  - Upload corpus to Pinecone")
        if not results["eval_data"]:
            print("  - Check evaluation dataset format")
        if not results["retrieval"]:
            print("  - Verify retriever functions")

    print()

    return all_passed, results


print(check_setup())

## Prompt

In [38]:
def create_rag_prompt(query: str, context: str, wikidata_info: str) -> str:
    """
    Creates a prompt for the LLM, incorporating retrieved context and Wikidata information.

    Args:
        query: The user's query.
        context: The retrieved context from the knowledge base.
        wikidata_info: Information retrieved from Wikidata.

    Returns:
        A formatted prompt string.
    """

    prompt = f"Context: {context}\nwiki: {wikidata_info}\nQ: {query}\nA:"

    return prompt

In [39]:
def generate_response(llm, prompt: str) -> str:
    """
    Generates a response from the LLM given a prompt.

    Args:
        llm: The language model object (e.g., ChatOpenAI).
        prompt: The prompt to send to the LLM.

    Returns:
        The generated response string.
    """

    messages = [HumanMessage(content=prompt)]
    response = llm.invoke(messages)

    return response.content

## Evaluation Pipeline

### Enhanced RAG Pipeline

In [40]:
def advanced_rag_pipeline(
    llm,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    query: str,
    retrieval_method: str = "hybrid",
    top_k: int = 10,
    alpha: float = 0.5,
    use_wikidata: bool = True,
    dynamic_threshold: bool = True,
) -> Tuple[str, dict]:
    """
    Performs Retrieval Augmented Generation (RAG) with hybrid retrieval,
    dynamic thresholding, and optional Wikidata integration.

    Args:
        llm: The language model to use for response generation.
        index: The Pinecone index for dense retrieval.
        bm25_index: The BM25Okapi index for sparse retrieval.
        corpus: The list of documents.
        query: The user's query.
        retrieval_method: The retrieval method to use ("dense", "sparse", or "hybrid").
        top_k: The number of documents to retrieve.
        alpha: Weight for dense retrieval in hybrid mode.
        use_wikidata: Whether to enrich context with Wikidata information.
        dynamic_threshold: Whether to use dynamic thresholding.

    Returns:
        A tuple containing:
            - The generated response string.
            - A dictionary of metadata (e.g., retrieval scores, threshold).
    """

    start_time = time.time()

    # 1. Retrieval
    if retrieval_method == "dense":
        retrieved_context = [
            hit["metadata"]["text"] for hit in retrieve_dense(index, query, top_k)
        ]
        threshold = None  # Not applicable for dense-only
    elif retrieval_method == "sparse":
        retrieved_context = [
            corpus[index]
            for index, score in retrieve_sparse(bm25_index, query, corpus, top_k)
        ]
        threshold = None  # Not applicable for sparse-only
    elif retrieval_method == "hybrid":
        retrieved_context, threshold = retrieve_hybrid(
            index, bm25_index, corpus, query, top_k, alpha, dynamic_threshold
        )
    else:
        raise ValueError(f"Invalid retrieval method: {retrieval_method}")

    context = "\n".join(retrieved_context) if retrieved_context else "No context found."

    # 2. Wikidata Enrichment
    wikidata_info = (
        enrich_text_with_wikidata(context)
        if use_wikidata and context != "No context found."
        else ""
    )

    # 3. Prompting and Generation
    prompt = create_rag_prompt(query, context, wikidata_info)
    response = generate_response(llm, prompt)

    end_time = time.time()

    metadata = {
        "retrieval_method": retrieval_method,
        "retrieved_documents": retrieved_context,
        "retrieved_docs": len(retrieved_context),
        "wikidata_entities": len(wikidata_info.split("\n")) if wikidata_info else 0,
        "response_tokens": count_tokens(response),
        "elapsed_time": end_time - start_time,
        "threshold": threshold,
    }

    return response, metadata

### Build Evaluation Dataset

In [41]:
def build_evaluation_dataset(
    llm,
    index: Pinecone.Index,
    bm25_index: BM25Okapi,
    corpus: List[str],
    dataframe: pd.DataFrame,
    retrieval_method: str,
    top_k: int,
    alpha: float,
    use_wikidata: bool,
    dynamic_threshold: bool,
) -> pd.DataFrame:
    """
    Builds a dataset for evaluating the RAG pipeline.

    Args:
        llm: The language model to use.
        index: The Pinecone index.
        bm25_index: The BM25 index.
        corpus: The list of documents.
        dataframe: A Pandas DataFrame containing questions and ground truth answers.
        retrieval_method: The retrieval method to use ("dense", "sparse", or "hybrid").
        top_k: The number of documents to retrieve.
        alpha: Weight for dense retrieval in hybrid mode.
        use_wikidata: Whether to use Wikidata.
        dynamic_threshold: Whether to use dynamic thresholding.

    Returns:
        A Pandas DataFrame with added 'answer' and 'contexts' columns.
    """

    answers = []
    contexts = []

    sampled_df = dataframe[:MAX_QUESTIONS].reset_index(
        drop=True
    )  # Limit to MAX_QUESTIONS
    questions = sampled_df["question"].tolist()  # Use the sampled questions

    for question in tqdm(questions, desc=f"Evaluating with {retrieval_method}"):
        try:
            response, metadata = advanced_rag_pipeline(
                llm=llm,
                index=index,
                bm25_index=bm25_index,
                corpus=corpus,
                query=question,
                retrieval_method=retrieval_method,
                top_k=top_k,
                alpha=alpha,
                use_wikidata=use_wikidata,
                dynamic_threshold=dynamic_threshold,
            )
            answers.append(response)
            contexts.append(metadata.get("retrieved_documents", []))
        except Exception as e:
            print(f"Error processing question: {question}. Error: {e}")
            answers.append(None)
            contexts.append(None)

    eval_df = sampled_df.copy()  # Use the sampled DataFrame for evaluation
    eval_df["answer"] = answers
    eval_df["contexts"] = contexts

    return eval_df

### RAGAS (Retrieval Augmented Generation Assessment)

In [42]:
def prepare_ragas_dataset(eval_df: pd.DataFrame) -> Dataset:
    """
    Prepares a Pandas DataFrame for evaluation with the Ragas library.

    Args:
        eval_df: A Pandas DataFrame containing 'question', 'ground_truth', 'answer',
                 and 'contexts' columns.

    Returns:
        A Hugging Face Dataset formatted for Ragas.
    """

    ragas_df = eval_df[["question", "ground_truth", "answer", "contexts"]].copy()

    # Ensure 'contexts' is a list of strings (Ragas requirement)
    def ensure_list_of_strings(example):
        if not isinstance(example["contexts"], list):
            example["contexts"] = [str(example["contexts"])]
        return example

    ragas_dataset = Dataset.from_pandas(ragas_df).map(ensure_list_of_strings)

    return ragas_dataset

### Pipeline

In [43]:
def evaluate_rag_system(
    llm_config: dict,
    retrieval_method: str,
    use_wikidata: bool,
    dynamic_threshold: bool,
) -> EvaluationResult:
    """
    Evaluates a RAG system using the Ragas library.

    Args:
        llm_config: A dictionary containing the configuration for the language model.
        retrieval_method: The retrieval method to use.
        use_wikidata: Whether to use Wikidata integration.
        dynamic_threshold: Whether to use dynamic thresholding.

    Returns:
        A Ragas evaluation result object containing the computed metrics.
    """

    eval_df = build_evaluation_dataset(
        **llm_config,
        retrieval_method=retrieval_method,
        use_wikidata=use_wikidata,
        dynamic_threshold=dynamic_threshold,
    )

    ragas_dataset = prepare_ragas_dataset(eval_df)

    result = evaluate(
        dataset=ragas_dataset,
        metrics=[
            answer_relevancy,
            answer_similarity,
            answer_correctness,
            factual_f1,
        ],
        raise_exceptions=False,
        llm=judge_llm,
        embeddings=embed_model,
        run_config=run_config,
    )

    return result

## Reporting

In [44]:
def report_ragas_evaluation_metrics(
    result_df: pd.DataFrame, metrics: List[str]
) -> dict:
    """
    Reports evaluation metrics from a Ragas evaluation result.

    Args:
        result_df: A Pandas DataFrame containing the evaluation results.
        metrics: A list of metric names to report.

    Returns:
        A dictionary containing the computed metrics.
    """

    evaluation = {}
    for metric in metrics:
        scores = result_df[metric].dropna().tolist()
        evaluation[f"{metric}_mean"] = result_df[metric].mean()
        evaluation[f"{metric}_std"] = result_df[metric].std()
        evaluation[f"{metric}_ci_low"] = np.percentile(scores, 2.5)
        evaluation[f"{metric}_ci_high"] = np.percentile(scores, 97.5)

        print(
            f"{metric}: {evaluation[f'{metric}_mean']:.3f} +- {evaluation[f'{metric}_std']:.3f} (95% CI: [{evaluation[f'{metric}_ci_low']:.3f}, {evaluation[f'{metric}_ci_high']:.3f}])"
        )

    return evaluation

In [45]:
def report(result: EvaluationResult, output: str = None) -> None:
    """
    Reports evaluation metrics from a Ragas evaluation result.

    Args:
        result: A Ragas evaluation result object.
        output: An optional path to save the evaluation results to a CSV file.
    """

    metrics = [
        "answer_relevancy",  # Relevancy of the answer to the question : 0 (not relevant) to 1 (highly relevant)
        "answer_similarity",  # Similarity of the answer to the question : 0 (not similar) to 1 (very similar)
        "answer_correctness",  # Correctness of the answer based on the ground truth : 0 (incorrect) to 1 (correct)
        "factual_correctness(mode=f1)",  # Factual correctness of the answer : 0 (completely incorrect) to 1 (completely correct)
    ]

    result_df = result.to_pandas()

    if output:
        result_df.to_csv(output, index=False)

    print()
    report_ragas_evaluation_metrics(result_df, metrics)

    print()
    for i in range(min(5, len(result_df))):
        print("Question:", result_df.iloc[i]["user_input"])
        print("Answer:", result_df.iloc[i]["response"])
        print("-----")

In [46]:
def perform_wilcoxon_test(
    enhanced: EvaluationResult, ablation: EvaluationResult
) -> None:
    """
    Performs a Wilcoxon signed-rank test to compare the performance of two methods.

    Args:
        enhanced: A Ragas evaluation result object for the enhanced method.
        ablation: A Ragas evaluation result object for the ablation method.
    """

    df_enhanced = enhanced.to_pandas()
    df_ablation = ablation.to_pandas()

    comparison_df = pd.merge(
        df_ablation[["user_input", "answer_correctness"]],
        df_enhanced[["user_input", "answer_correctness"]],
        on="user_input",
        suffixes=("_ablation", "_enhanced"),
    ).dropna()

    stat, p_value = wilcoxon(
        comparison_df["answer_correctness_enhanced"],
        comparison_df["answer_correctness_ablation"],
        alternative="greater",
    )

    print(f"Wilcoxon signed-rank test statistic: {stat}, p-value: {p_value}")

    if p_value < 0.05:
        print("The enhanced method is significantly better than the ablation method.")
    else:
        print("No significant difference between the enhanced and ablation methods.")

    wins_enhanced = (
        comparison_df["answer_correctness_enhanced"]
        > comparison_df["answer_correctness_ablation"]
    ).sum()
    wins_ablation = (
        comparison_df["answer_correctness_enhanced"]
        < comparison_df["answer_correctness_ablation"]
    ).sum()
    ties = (
        comparison_df["answer_correctness_enhanced"]
        == comparison_df["answer_correctness_ablation"]
    ).sum()

    print(f"Enhanced Wins: {wins_enhanced}")
    print(f"Ablation Wins: {wins_ablation}")
    print(f"Ties: {ties}")

## Human Evaluation

In [47]:
def create_human_evaluation_sample(
    result_df: pd.DataFrame, n: int = 30
) -> pd.DataFrame:
    """
    Creates a human evaluation sample from a Ragas evaluation result.

    Args:
        result_df: A Pandas DataFrame containing the evaluation results.
        n: The number of samples to create.

    Returns:
        A Pandas DataFrame containing the human evaluation sample.
    """

    scores = result_df["answer_correctness"].values

    q1, q2, q3 = np.percentile(scores, [25, 50, 75])

    low = result_df[scores < q1].head(n // 4)
    mid_low = result_df[(scores >= q1) & (scores < q2)].head(n // 4)
    mid_high = result_df[(scores >= q2) & (scores < q3)].head(n // 4)
    high = result_df[scores >= q3].head(n // 4)

    sample = pd.concat([low, mid_low, mid_high, high])

    verification = []
    for idx, row in sample.iterrows():
        verification.append(
            {
                "id": idx,
                "question": row["user_input"],
                "generated_answer": row["response"],
                "ground_truth": row["reference"],
                "ragas_score": row["answer_correctness"],
                "human_score": "",
                "gpt_score": "",
            }
        )

    df = pd.DataFrame(verification)
    df.to_csv("human_verification.csv", index=False)

    print(f"Created {len(df)} samples for verification")
    print(f"Quartiles: Q1={q1:.3f}, Q2={q2:.3f}, Q3={q3:.3f}")

    return df

In [48]:
def gpt_annotate(question: str, answer: str, ground_truth: str) -> int:
    """
    Annotates the correctness of an answer based on a question and ground truth.

    Args:
        question: The question.
        answer: The answer.
        ground_truth: The ground truth.

    Returns:
        The correctness of the answer.
    """

    prompt = f"""Rate correctness 0-2:
Q: {question}
GT: {ground_truth}
A: {answer}
Return only 0,1,2:"""
    response = judge_llm.invoke([HumanMessage(content=prompt)])
    return int(response.content.strip())

In [49]:
def analyze_agreement(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    human = df["human_score"].values
    gpt = df["gpt_score"].values

    # Exact agreement
    exact = (human == gpt).sum()
    exact_pct = exact / len(df) * 100

    # Cohen's Kappa
    from sklearn.metrics import cohen_kappa_score

    kappa = cohen_kappa_score(human, gpt)

    # Correlation with RAGAS
    from scipy.stats import pearsonr

    corr, p = pearsonr(human, df["ragas_score"])

    print(f"Sample: {len(df)}")
    print(f"Exact agreement: {exact}/{len(df)} ({exact_pct:.1f}%)")
    print(f"Cohen's Kappa: {kappa:.3f}")
    print(f"Human vs RAGAS correlation: {corr:.3f} (p={p:.4f})")

    return df

## GPT-4 (gpt-4o-mini)

In [50]:
chat_gpt_4_model = "gpt-4o-mini"

chat_gpt_4_llm = ChatOpenAI(
    model_name=chat_gpt_4_model,
    openai_api_key=os.environ["OPENAI_API_KEY"],
    max_tokens=MAX_TOKENS,
)

chat_gpt_4_llm_config = {
    "llm": chat_gpt_4_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": TOP_K,
    "alpha": ALPHA,
}

### Enhanced RAG

In [ ]:
chat_gpt_4_result_hybrid = evaluate_rag_system(
    llm_config=chat_gpt_4_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(chat_gpt_4_result_hybrid, "chat_gpt_4_hybrid_results.csv")

In [ ]:
human_sample = create_human_evaluation_sample(chat_gpt_4_result_hybrid.to_pandas())

In [ ]:
gpt_scores = []
for idx, row in tqdm(human_sample.iterrows(), total=len(human_sample)):
    score = gpt_annotate(row["question"], row["generated_answer"], row["ground_truth"])
    gpt_scores.append(score)
    time.sleep(0.3)

human_sample["gpt_score"] = gpt_scores
human_sample.to_csv("human_verification_with_gpt.csv", index=False)
print("GPT annotation done!")

In [ ]:
if os.path.exists("human_verification_complete.csv"):
    results = analyze_agreement("human_verification_complete.csv")
else:
    print("Please complete manual annotation first!")

### Ablation : No Hybrid Retrieval

In [ ]:
chat_gpt_4_result_dense = evaluate_rag_system(
    llm_config=chat_gpt_4_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(chat_gpt_4_result_dense, "chat_gpt_4_dense_results.csv")

In [ ]:
perform_wilcoxon_test(chat_gpt_4_result_hybrid, chat_gpt_4_result_dense)

### Ablation : No Wikidata

In [ ]:
chat_gpt_4_result_hybrid_no_wikidata = evaluate_rag_system(
    llm_config=chat_gpt_4_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)

In [ ]:
report(
    chat_gpt_4_result_hybrid_no_wikidata, "chat_gpt_4_hybrid_no_wikidata_results.csv"
)

In [ ]:
perform_wilcoxon_test(chat_gpt_4_result_hybrid, chat_gpt_4_result_hybrid_no_wikidata)

### Ablation : Fixed Threshold

In [ ]:
chat_gpt_4_result_hybrid_fixed_threshold = evaluate_rag_system(
    llm_config=chat_gpt_4_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

In [ ]:
report(
    chat_gpt_4_result_hybrid_fixed_threshold,
    "chat_gpt_4_hybrid_fixed_threshold_results.csv",
)

In [ ]:
perform_wilcoxon_test(
    chat_gpt_4_result_hybrid, chat_gpt_4_result_hybrid_fixed_threshold
)

## GPT-3.5 (gpt-3.5-turbo)

In [ ]:
chat_gpt_3_model = "gpt-3.5-turbo"

chat_gpt_3_llm = ChatOpenAI(
    model_name=chat_gpt_3_model,
    openai_api_key=os.environ["OPENAI_API_KEY"],
    max_tokens=MAX_TOKENS,
)

chat_gpt_3_llm_config = {
    "llm": chat_gpt_3_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": TOP_K,
    "alpha": ALPHA,
}

### Enhanced RAG

In [ ]:
chat_gpt_3_result_hybrid = evaluate_rag_system(
    llm_config=chat_gpt_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(chat_gpt_3_result_hybrid, "chat_gpt_3_hybrid_results.csv")

### Ablation : No Hybrid Retrieval

In [ ]:
chat_gpt_3_result_dense = evaluate_rag_system(
    llm_config=chat_gpt_3_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,  # Will be ignored
)

In [ ]:
report(chat_gpt_3_result_dense, "chat_gpt_3_dense_results.csv")

In [ ]:
perform_wilcoxon_test(chat_gpt_3_result_hybrid, chat_gpt_3_result_dense)

### Ablation : No Wikidata

In [ ]:
chat_gpt_3_result_hybrid_no_wikidata = evaluate_rag_system(
    llm_config=chat_gpt_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)

In [ ]:
report(
    chat_gpt_3_result_hybrid_no_wikidata, "chat_gpt_3_hybrid_no_wikidata_results.csv"
)

In [ ]:
perform_wilcoxon_test(chat_gpt_3_result_hybrid, chat_gpt_3_result_hybrid_no_wikidata)

### Ablation : Fixed Threshold

In [ ]:
chat_gpt_3_result_hybrid_fixed_threshold = evaluate_rag_system(
    llm_config=chat_gpt_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

In [ ]:
report(
    chat_gpt_3_result_hybrid_fixed_threshold,
    "chat_gpt_3_hybrid_fixed_threshold_results.csv",
)

In [ ]:
perform_wilcoxon_test(
    chat_gpt_3_result_hybrid, chat_gpt_3_result_hybrid_fixed_threshold
)

## Mistral (mistralai/Mistral-7B-Instruct-v0.1)

In [ ]:
mistral_model = "mistralai/Mistral-7B-Instruct-v0.1"

mistral_llm = ChatOpenAI(
    model_name=mistral_model,
    openai_api_key=DEEP_INFRA_API_KEY,
    openai_api_base=DEEP_INFRA_API_BASE,
    max_tokens=MAX_TOKENS,
)

mistral_llm_config = {
    "llm": mistral_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": TOP_K,
    "alpha": ALPHA,
}

### Enhanced RAG

In [ ]:
mistral_result_hybrid = evaluate_rag_system(
    llm_config=mistral_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(mistral_result_hybrid, "mistral_hybrid_results.csv")

### Ablation : No Hybrid Retrieval

In [ ]:
mistral_result_dense = evaluate_rag_system(
    llm_config=mistral_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(mistral_result_dense, "mistral_dense_results.csv")

In [ ]:
perform_wilcoxon_test(mistral_result_hybrid, mistral_result_dense)

### Ablation : No Wikidata

In [ ]:
mistral_result_hybrid_no_wikidata = evaluate_rag_system(
    llm_config=mistral_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)

In [ ]:
report(mistral_result_hybrid_no_wikidata, "mistral_hybrid_no_wikidata_results.csv")

In [ ]:
perform_wilcoxon_test(mistral_result_hybrid, mistral_result_hybrid_no_wikidata)

### Ablation : Fixed Threshold

In [ ]:
mistral_result_hybrid_fixed_threshold = evaluate_rag_system(
    llm_config=mistral_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

In [ ]:
report(
    mistral_result_hybrid_fixed_threshold, "mistral_hybrid_fixed_threshold_results.csv"
)

In [ ]:
perform_wilcoxon_test(mistral_result_hybrid, mistral_result_hybrid_fixed_threshold)

## Llama 3 (meta-llama/Llama-3.3-70B-Instruct-Turbo)

In [ ]:
llama_3_model = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

llama_3_llm = ChatOpenAI(
    model_name=llama_3_model,
    openai_api_key=DEEP_INFRA_API_KEY,
    openai_api_base=DEEP_INFRA_API_BASE,
    max_tokens=MAX_TOKENS,
)

llama_3_llm_config = {
    "llm": llama_3_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": TOP_K,
    "alpha": ALPHA,
}

### Enhanced RAG

In [ ]:
llama_3_result_hybrid = evaluate_rag_system(
    llm_config=llama_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(llama_3_result_hybrid, "llama_3_hybrid_results.csv")

### Ablation : No Hybrid Retrieval

In [ ]:
llama_3_result_dense = evaluate_rag_system(
    llm_config=llama_3_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(llama_3_result_dense, "llama_3_dense_results.csv")

In [ ]:
perform_wilcoxon_test(llama_3_result_hybrid, llama_3_result_dense)

### Ablation : No Wikidata

In [ ]:
llama_3_result_hybrid_no_wikidata = evaluate_rag_system(
    llm_config=llama_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)

In [ ]:
report(llama_3_result_hybrid_no_wikidata, "llama_3_hybrid_no_wikidata_results.csv")

In [ ]:
perform_wilcoxon_test(llama_3_result_hybrid, llama_3_result_hybrid_no_wikidata)

### Ablation : Fixed Threshold

In [ ]:
llama_3_result_hybrid_fixed_threshold = evaluate_rag_system(
    llm_config=llama_3_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

In [ ]:
report(
    llama_3_result_hybrid_fixed_threshold, "llama_3_hybrid_fixed_threshold_results.csv"
)

In [ ]:
perform_wilcoxon_test(llama_3_result_hybrid, llama_3_result_hybrid_fixed_threshold)

## Llama 4 (meta-llama/Llama-4-Scout-17B-16E-Instruct)

In [ ]:
llama_4_model = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

llama_4_llm = ChatOpenAI(
    model_name=llama_4_model,
    openai_api_key=DEEP_INFRA_API_KEY,
    openai_api_base=DEEP_INFRA_API_BASE,
    max_tokens=MAX_TOKENS,
)

llama_4_llm_config = {
    "llm": llama_4_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": TOP_K,
    "alpha": ALPHA,
}

### Enhanced RAG

In [ ]:
llama_4_result_hybrid = evaluate_rag_system(
    llm_config=llama_4_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(llama_4_result_hybrid, "llama_4_hybrid_results.csv")

### Ablation : No Hybrid Retrieval

In [ ]:
llama_4_result_dense = evaluate_rag_system(
    llm_config=llama_4_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(llama_4_result_dense, "llama_4_dense_results.csv")

In [ ]:
perform_wilcoxon_test(llama_4_result_hybrid, llama_4_result_dense)

### Ablation : No Wikidata

In [ ]:
llama_4_result_hybrid_no_wikidata = evaluate_rag_system(
    llm_config=llama_4_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)

In [ ]:
report(llama_4_result_hybrid_no_wikidata, "llama_4_hybrid_no_wikidata_results.csv")

In [ ]:
perform_wilcoxon_test(llama_4_result_hybrid, llama_4_result_hybrid_no_wikidata)

### Ablation : Fixed Threshold

In [ ]:
llama_4_result_hybrid_fixed_threshold = evaluate_rag_system(
    llm_config=llama_4_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

In [ ]:
report(
    llama_4_result_hybrid_fixed_threshold, "llama_4_hybrid_fixed_threshold_results.csv"
)

In [ ]:
perform_wilcoxon_test(llama_4_result_hybrid, llama_4_result_hybrid_fixed_threshold)

## DeepSeek (deepseek-ai/DeepSeek-V3)

In [ ]:
deepseek_model = "deepseek-ai/DeepSeek-V3"

deepseek_llm = ChatOpenAI(
    model_name=deepseek_model,
    openai_api_key=DEEP_INFRA_API_KEY,
    openai_api_base=DEEP_INFRA_API_BASE,
    max_tokens=MAX_TOKENS,
)

deepseek_llm_config = {
    "llm": deepseek_llm,
    "index": index,
    "bm25_index": bm25_index,
    "corpus": corpus,
    "dataframe": eval_df_limited,
    "top_k": 10,
    "alpha": 0.6,
}

### Enhanced RAG

In [ ]:
deepseek_result_hybrid = evaluate_rag_system(
    llm_config=deepseek_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(deepseek_result_hybrid, "deepseek_hybrid_results.csv")

### Ablation : No Hybrid Retrieval

In [ ]:
deepseek_result_dense = evaluate_rag_system(
    llm_config=deepseek_llm_config,
    retrieval_method="dense",
    use_wikidata=True,
    dynamic_threshold=True,
)

In [ ]:
report(deepseek_result_dense, "deepseek_dense_results.csv")

In [ ]:
perform_wilcoxon_test(deepseek_result_hybrid, deepseek_result_dense)

### Ablation : No Wikidata

In [ ]:
deepseek_result_hybrid_no_wikidata = evaluate_rag_system(
    llm_config=deepseek_llm_config,
    retrieval_method="hybrid",
    use_wikidata=False,
    dynamic_threshold=True,
)

In [ ]:
report(deepseek_result_hybrid_no_wikidata, "deepseek_hybrid_no_wikidata_results.csv")

In [ ]:
perform_wilcoxon_test(deepseek_result_hybrid, deepseek_result_hybrid_no_wikidata)

### Ablation : Fixed Threshold

In [ ]:
deepseek_result_hybrid_fixed_threshold = evaluate_rag_system(
    llm_config=deepseek_llm_config,
    retrieval_method="hybrid",
    use_wikidata=True,
    dynamic_threshold=False,
)

In [ ]:
report(
    deepseek_result_hybrid_fixed_threshold,
    "deepseek_hybrid_fixed_threshold_results.csv",
)

In [ ]:
perform_wilcoxon_test(deepseek_result_hybrid, deepseek_result_hybrid_fixed_threshold)